In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable GPU in Colab runtime settings.")

CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Week8"
ADAPTER_DIR = os.path.join(BASE_DIR, "adapters")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("ADAPTER_DIR:", ADAPTER_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("train exists:", os.path.exists(os.path.join(BASE_DIR, "train.jsonl")))
print("val exists  :", os.path.exists(os.path.join(BASE_DIR, "val.jsonl")))

ADAPTER_DIR: /content/drive/MyDrive/Colab Notebooks/Week8/adapters
RESULTS_DIR: /content/drive/MyDrive/Colab Notebooks/Week8/results
train exists: True
val exists  : True


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": os.path.join(BASE_DIR, "train.jsonl"),
        "validation": os.path.join(BASE_DIR, "val.jsonl")
    }
)

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 990
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 110
    })
})
{'instruction': 'Answer the healthcare question clearly and safely.', 'input': 'Give a simple explanation of palpitations.', 'output': 'Palpitations are feelings of a fast, pounding, or irregular heartbeat. It may have many possible causes depending on the overall health context.'}


In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pad token: <|endoftext|>
EOS token: <|im_end|>


In [11]:
MAX_LENGTH = 256  # memory-saving trick for Colab stability

def tokenize_function(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""
    answer = example["output"]
    full_text = prompt + answer

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokenized_prompt = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]
    labels = input_ids.copy()

    prompt_len = len(tokenized_prompt["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0].keys())

Map:   0%|          | 0/990 [00:00<?, ? examples/s]

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 990
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 110
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [12]:
sample = tokenized_dataset["train"][0]
print("First 60 labels:")
print(sample["labels"][:60])

First 60 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 19980, 79, 30667, 525, 15650, 315, 264, 4937, 11, 69527, 11, 476, 41308, 52105, 13, 1084, 1231, 614, 1657, 3204, 11137, 11649, 389, 279, 8084, 2820, 2266, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643]


In [13]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
print(type(model))

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [14]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
print(type(model))

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print(type(model))
model.print_trainable_parameters()

<class 'peft.peft_model.PeftModelForCausalLM'>
trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [16]:

trainable = 0
total = 0

for _, param in model.named_parameters():
    total += param.numel()
    if param.requires_grad:
        trainable += param.numel()

print(f"Trainable params: {trainable}")
print(f"Total params: {total}")
print(f"Trainable %: {100 * trainable / total:.4f}")
print("Target: approximately ~1%")

Trainable params: 2179072
Total params: 890795520
Trainable %: 0.2446
Target: approximately ~1%


In [17]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=RESULTS_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=50,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss,Validation Loss
50,1.292064,1.177572
100,0.734790,0.694116
150,0.474736,0.559106
200,0.456609,0.467322
250,0.446974,0.427989
300,0.367302,0.389980
350,0.287246,0.365143
400,0.287907,0.345722
450,0.281896,0.337938
500,0.299929,0.319947


TrainOutput(global_step=744, training_loss=0.48169687390327454, metrics={'train_runtime': 815.9631, 'train_samples_per_second': 3.64, 'train_steps_per_second': 0.912, 'total_flos': 5987609778585600.0, 'train_loss': 0.48169687390327454, 'epoch': 3.0})

In [25]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

('/content/drive/MyDrive/Colab Notebooks/Week8/adapters/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Week8/adapters/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/Week8/adapters/tokenizer.json')

In [22]:
model.eval()

prompt = """### Instruction:
Analyze the symptoms and provide a safe, non-diagnostic explanation.

### Input:
A patient has fatigue, increased thirst, and frequent urination.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Instruction:
Analyze the symptoms and provide a safe, non-diagnostic explanation.

### Input:
A patient has fatigue, increased thirst, and frequent urination.

### Response:
Possible Association: The symptom pattern of fatigue, increased thirst, frequent urination may be associated with dehydration or high blood sugar, infection, anemia, or another underlying condition. Concern Level: Low to moderate. Advice: This is not enough to make a diagnosis, but a healthcare professional can assess the symptoms properly.


In [26]:
import shutil

shutil.make_archive("adapters_zip", "zip", ADAPTER_DIR)

'/content/adapters_zip.zip'

In [27]:
from google.colab import files

files.download("adapters_zip.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>